# Preprocessing diabetes data

##  Import libraries

In [13]:
!pip install pandas
!pip install numpy
!pip install scikit-learn

In [14]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import os
import glob

## Code mapping

In [15]:
CODE_LABELS = {
    33: "insulin_regular",
    34: "insulin_nph",
    35: "insulin_ultralente",
    48: "glucose_unspecified",
    57: "glucose_unspecified2",
    58: "glucose_pre_breakfast",
    59: "glucose_post_breakfast",
    60: "glucose_pre_lunch",
    61: "glucose_post_lunch",
    62: "glucose_pre_supper",
    63: "glucose_post_supper",
    64: "glucose_pre_snack",
    65: "hypoglycemic_symptoms",
    66: "meal_typical",
    67: "meal_more_than_usual",
    68: "meal_less_than_usual",
    69: "exercise_typical",
    70: "exercise_more_than_usual",
    71: "exercise_less_than_usual",
    72: "special_event",
}
 
GLUCOSE_CODES  = {48, 57, 58, 59, 60, 61, 62, 63, 64}
INSULIN_CODES  = {33, 34, 35}
MEAL_CODES     = {66, 67, 68}
EXERCISE_CODES = {69, 70, 71}


## Load raw data

In [16]:
DATA_DIR = "../raw/diabetes/diabetes-data/Diabetes-Data"  
 
def load_patients(data_dir: str) -> pd.DataFrame:
    """Read per-patient files and concatenate into a single DataFrame."""
    files = sorted(glob.glob(os.path.join(data_dir, "data-*"))) 
    frames = []
    for path in files:
        patient_id = int(os.path.basename(path).split("-")[1])
        df = pd.read_csv(
            path,
            sep="\t",
            header=None,
            names=["date", "time", "code", "value"],
        )
        df["patient_id"] = patient_id
        frames.append(df)
    return pd.concat(frames, ignore_index=True)
 
raw = load_patients(DATA_DIR)
print(f"Raw shape: {raw.shape}")


Raw shape: (29330, 5)


## Basic cleaning

In [17]:
# Keep only known integer codes (drop stray strings like '3A', '22', '21', etc.)
raw["code"] = pd.to_numeric(raw["code"], errors="coerce")
raw = raw.dropna(subset=["code"])
raw["code"] = raw["code"].astype(int)
raw = raw[raw["code"].isin(CODE_LABELS)]
 
# Parse datetime
raw["datetime"] = pd.to_datetime(
    raw["date"] + " " + raw["time"], format="%m-%d-%Y %H:%M", errors="coerce"
)
raw = raw.dropna(subset=["datetime"])
raw = raw.drop(columns=["date", "time"])
 
# Numeric value (some entries may be missing / non-numeric)
raw["value"] = pd.to_numeric(raw["value"], errors="coerce")

 
print(f"After cleaning: {raw.shape}")


After cleaning: (29131, 4)


## One row per patient

In [18]:
raw["day"] = raw["datetime"].dt.date
 
def daily_features(group: pd.DataFrame) -> pd.Series:
    """Compute one day's feature vector for a single patient."""
    row = {}
 
    # Insulin totals
    for code, name in [(33, "regular"), (34, "nph"), (35, "ultralente")]:
        doses = group.loc[group["code"] == code, "value"].dropna()
        row[f"insulin_{name}_total"] = doses.sum()
        row[f"insulin_{name}_doses"] = doses.count()
 
    # Glucose stats (across all glucose measurement types)
    glucose = group.loc[group["code"].isin(GLUCOSE_CODES), "value"].dropna()
    row["glucose_mean"]  = glucose.mean()   if len(glucose) else np.nan
    row["glucose_min"]   = glucose.min()    if len(glucose) else np.nan
    row["glucose_max"]   = glucose.max()    if len(glucose) else np.nan
    row["glucose_std"]   = glucose.std()    if len(glucose) else np.nan
    row["glucose_count"] = glucose.count()
 
    # Hypoglycemic events
    row["hypoglycemic_events"] = (group["code"] == 65).sum()
 
    # Meal patterns (encode as ordinal: 0=less, 1=typical, 2=more)
    meal_map = {68: 0, 66: 1, 67: 2}
    meal_codes = group.loc[group["code"].isin(MEAL_CODES), "code"]
    row["meal_pattern"] = meal_codes.map(meal_map).mean() if len(meal_codes) else np.nan
 
    # Exercise patterns (same ordinal scheme)
    exercise_map = {71: 0, 69: 1, 70: 2}
    exercise_codes = group.loc[group["code"].isin(EXERCISE_CODES), "code"]
    row["exercise_pattern"] = exercise_codes.map(exercise_map).mean() if len(exercise_codes) else np.nan
 
    return pd.Series(row)
 
daily = (
    raw.groupby(["patient_id", "day"])
    .apply(daily_features)
    .reset_index()
)
 
print(f"Daily feature matrix shape: {daily.shape}")


Daily feature matrix shape: (3880, 16)


/tmp/ipykernel_22445/969470795.py:38: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(daily_features)


## Target

In [19]:
daily = daily.sort_values(["patient_id", "day"])
next_day_events = daily.groupby("patient_id")["hypoglycemic_events"].shift(-1)
daily["hypo_next_day"] = (
    next_day_events.gt(0)
    .where(next_day_events.notna())
)
 
# Drop rows with no target (last day of each patient)
daily = daily.dropna(subset=["hypo_next_day"])
daily["hypo_next_day"] = daily["hypo_next_day"].astype(int)


## Train/Test split

In [20]:
X = daily.drop(columns=["patient_id", "day", "hypo_next_day"])
y = daily["hypo_next_day"].rename("target")
groups = daily["patient_id"]

# Keep every record from one patient in exactly one split.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

train_patients = set(groups.iloc[train_idx])
test_patients = set(groups.iloc[test_idx])
assert train_patients.isdisjoint(test_patients)
 


## Missing values

In [21]:
# Columns that are counts → 0 when absent
count_cols = [
    "insulin_regular_doses", "insulin_nph_doses", "insulin_ultralente_doses",
    "glucose_count", "hypoglycemic_events",
]
# Columns that are sums → 0 when absent
sum_cols = [
    "insulin_regular_total", "insulin_nph_total", "insulin_ultralente_total",
]
# Columns that are means/stats → impute with training mean
stat_cols = [
    "glucose_mean", "glucose_min", "glucose_max", "glucose_std",
    "meal_pattern", "exercise_pattern",
]
 
for col in count_cols + sum_cols:
    X_train[col] = X_train[col].fillna(0)
    X_test[col]  = X_test[col].fillna(0)
 
for col in stat_cols:
    mean_val = X_train[col].mean()
    X_train[col] = X_train[col].fillna(mean_val)
    X_test[col]  = X_test[col].fillna(mean_val)


## Normalization

In [22]:
# patient_id and day were removed before splitting; export the remaining
# imputed features before normalization for TabLLM.
serialization_output_dir = Path("../pre-processed/serialization")
serialization_output_dir.mkdir(parents=True, exist_ok=True)
pd.concat([X_train.copy(), y_train], axis=1).to_csv(
    serialization_output_dir / "diabetes_train.csv", index=False
)
pd.concat([X_test.copy(), y_test], axis=1).to_csv(
    serialization_output_dir / "diabetes_test.csv", index=False
)

numerical_columns = list(X_train.columns)   # all features are numerical here
 
# Fit normalization only on training data, then apply it to test data.
scaler = StandardScaler()
X_train[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_test[numerical_columns]  = scaler.transform(X_test[numerical_columns])


## Export

In [23]:
train_data = pd.concat([X_train, y_train], axis=1)
test_data  = pd.concat([X_test,  y_test],  axis=1)
 
disease_name = "diabetes"

print(f"\nExported → {disease_name}_train.csv ({train_data.shape}), {disease_name}_test.csv ({test_data.shape})")
print(f"Target balance (train): {y_train.value_counts().to_dict()}")
print("\nFeature columns:", list(X_train.columns))



Exported → diabetes_train.csv ((2989, 15)), diabetes_test.csv ((821, 15))
Target balance (train): {0: 2801, 1: 188}

Feature columns: ['insulin_regular_total', 'insulin_regular_doses', 'insulin_nph_total', 'insulin_nph_doses', 'insulin_ultralente_total', 'insulin_ultralente_doses', 'glucose_mean', 'glucose_min', 'glucose_max', 'glucose_std', 'glucose_count', 'hypoglycemic_events', 'meal_pattern', 'exercise_pattern']


In [24]:
output_dir = Path("../pre-processed")
output_dir.mkdir(parents=True, exist_ok=True)

train_data.to_csv(output_dir / f"{disease_name}_train.csv", index=False)
test_data.to_csv(output_dir / f"{disease_name}_test.csv", index=False)